# WP1 Tourism Heat Exposure Data Cube

## CDS STAC + ERA5-HEAT UTCI + OSM Amenities + DGGS + Zarr

This notebook is a proposal-ready and implementation-ready workflow for **WP1: Modelling heat hazards and tourism exposure**.

It shows how to connect:

1. **Thermal comfort indices derived from ERA5 reanalysis** through the CDS STAC catalogue.
2. **Tourism exposure layers** from OpenStreetMap using the Overpass API.
3. **DGGS / H3 cells** as the common grid for joining raster climate data and vector tourism assets.
4. **Zarr data cubes** for scalable storage, querying, and reuse.
5. **STAC metadata** for publishing derived tourism heat exposure outputs.

The workflow is designed for global Top 100 tourism city screening and detailed case-study cities such as **Brisbane, Gold Coast, and Sydney**.

## 1. Why this architecture helps WP1

The project combines datasets with very different structures:

| Dataset | Format | Spatial structure | Challenge |
|---|---|---|---|
| ERA5-HEAT / UTCI | gridded climate data | raster/time cube | large time series, climate thresholds |
| NASA downscaled projections | gridded climate data | raster/time/scenario cube | future periods and scenarios |
| Bureau of Meteorology observations | station data | points | local validation against gridded data |
| Landsat 8 urban heat maps | high-resolution raster | raster scenes | intra-urban surface heat |
| OSM tourism amenities | vector data | points, lines, polygons | hotels, restaurants, museums, parks, transport |
| smartphone thermal observations | field observations | points/time | tourism micro-climates |

A **DGGS cell ID**, for example an H3 index, becomes the shared spatial key:

```text
climate raster pixel -> DGGS cell
Landsat LST pixel    -> DGGS cell
hotel / museum / POI -> DGGS cell
thermal camera point -> DGGS cell
transport corridor   -> DGGS cells
```

After that, the analysis becomes a table join instead of repeated raster-vector overlay.

## 2. Proposed pipeline

```text
CDS STAC catalogue
    |
    |-- ERA5-HEAT UTCI / MRT metadata and retrieval links
    |-- optional ARCO/Zarr assets if available
    v
xarray + dask climate cube
    |
    |-- UTCI thresholds
    |-- heat stress days
    |-- seasonal and annual summaries
    v
DGGS / H3 climate cell table
    |
    |-- h3 cell id
    |-- city
    |-- year / period / scenario
    |-- utci_days_above_32
    |-- utci_days_above_38
    v
OSM Overpass tourism layers
    |
    |-- hotels and accommodation
    |-- restaurants and cafes
    |-- museums and attractions
    |-- parks, beaches, outdoor sites
    |-- transport stations and corridors
    v
DGGS exposure table
    |
    |-- h3 cell id
    |-- accommodation_count
    |-- food_drink_count
    |-- attraction_count
    |-- transport_count
    |-- outdoor_count
    v
Tourism heat exposure cube
    |
    |-- Zarr for gridded/time variables
    |-- GeoParquet for DGGS vector tables
    |-- STAC metadata for discovery/query
```

## 3. Environment setup

Install the packages below in a clean conda or virtual environment.

```bash
pip install requests pandas geopandas shapely pyarrow matplotlib xarray dask zarr netcdf4 h3 pystac pystac-client folium
```

Optional for raster and cloud workflows:

```bash
pip install rioxarray rasterio odc-stac stackstac exactextract xvec cdsapi
```

In [ ]:
import json
import time
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString, box
import matplotlib.pyplot as plt

import xarray as xr

# DGGS / H3
import h3

# STAC helpers
from pystac_client import Client

## 4. Project parameters

The bounding boxes are in `west, south, east, north` order.  
Overpass uses `south, west, north, east`, so a helper function converts the order later.

In [ ]:
PROJECT_DIR = Path("tourism_heat_cube")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
STAC_DIR = PROJECT_DIR / "stac"

for p in [DATA_DIR, OUTPUT_DIR, STAC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Case-study city bounding boxes: west, south, east, north
CITY_BBOXES = {
    "Brisbane": (152.75, -27.75, 153.35, -27.15),
    "Gold_Coast": (153.20, -28.25, 153.65, -27.70),
    "Sydney": (150.55, -34.20, 151.35, -33.55),
}

# H3 resolutions
H3_GLOBAL_RES = 5      # global Top 100 tourism city comparison
H3_CITY_RES = 7        # city-wide exposure summaries
H3_LOCAL_RES = 9       # site/neighbourhood tourism micro-climate analysis

# Heat stress thresholds in Celsius for UTCI
UTCI_STRONG_HEAT = 32
UTCI_VERY_STRONG_HEAT = 38

## 5. Connect to the CDS STAC catalogue

The CDS catalogue URL provided for this project is:

```text
https://cds.climate.copernicus.eu/api/catalogue/v1/collections/derived-utci-historical
```

This collection represents **Thermal comfort indices derived from ERA5 reanalysis**, also known as **ERA5-HEAT**.

The important variables for this WP are:

- `universal_thermal_climate_index` / UTCI
- `mean_radiant_temperature` / MRT

The code below reads the STAC collection metadata directly from the CDS catalogue.

In [ ]:
CDS_STAC_ROOT = "https://cds.climate.copernicus.eu/api/catalogue/v1/"
CDS_UTCI_COLLECTION_URL = "https://cds.climate.copernicus.eu/api/catalogue/v1/collections/derived-utci-historical"

response = requests.get(CDS_UTCI_COLLECTION_URL, timeout=60)
response.raise_for_status()
utci_collection = response.json()

print("Collection ID:", utci_collection.get("id"))
print("Title:", utci_collection.get("title"))
print("Temporal extent:", utci_collection.get("extent", {}).get("temporal", {}).get("interval"))
print("Spatial bbox:", utci_collection.get("extent", {}).get("spatial", {}).get("bbox"))
print("Updated:", utci_collection.get("updated"))
print("Available top-level keys:", list(utci_collection.keys()))

In [ ]:
# Extract useful CDS links: retrieve endpoint, form, constraints, related time-series collection.
links = utci_collection.get("links", [])

links_df = pd.DataFrame([
    {
        "rel": link.get("rel"),
        "title": link.get("title"),
        "type": link.get("type"),
        "href": link.get("href"),
    }
    for link in links
])

links_df.head(20)

In [ ]:
def get_first_link(collection: Dict, rel: str, title_contains: Optional[str] = None) -> Optional[str]:
    """Return the first link href matching rel and optional title text."""
    for link in collection.get("links", []):
        if link.get("rel") != rel:
            continue
        if title_contains is not None:
            title = (link.get("title") or "").lower()
            if title_contains.lower() not in title:
                continue
        return link.get("href")
    return None

retrieve_href = get_first_link(utci_collection, "retrieve")
form_href = get_first_link(utci_collection, "form")
constraints_href = get_first_link(utci_collection, "constraints")
timeseries_collection_href = get_first_link(utci_collection, "related", "time-series")

print("Retrieve endpoint:", retrieve_href)
print("Form metadata:", form_href)
print("Constraints metadata:", constraints_href)
print("Related time-series collection:", timeseries_collection_href)

### 5.1 Inspect request form and constraints

CDS datasets may change request fields over time.  
For a robust pipeline, read the current **form** and **constraints** from the STAC collection instead of hard-coding all parameters.

In [ ]:
def safe_get_json(url: Optional[str]) -> Optional[Dict]:
    if not url:
        return None
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

form_json = safe_get_json(form_href)
constraints_json = safe_get_json(constraints_href)

print("Form keys:", list(form_json.keys()) if isinstance(form_json, dict) else None)
print("Constraints keys:", list(constraints_json.keys()) if isinstance(constraints_json, dict) else None)

# Save metadata locally for reproducibility.
with open(DATA_DIR / "cds_derived_utci_historical_collection.json", "w") as f:
    json.dump(utci_collection, f, indent=2)

if form_json:
    with open(DATA_DIR / "cds_derived_utci_historical_form.json", "w") as f:
        json.dump(form_json, f, indent=2)

if constraints_json:
    with open(DATA_DIR / "cds_derived_utci_historical_constraints.json", "w") as f:
        json.dump(constraints_json, f, indent=2)

## 6. Option A: Use CDS API retrieval for a city and period

The STAC collection exposes a `retrieve` link. For reproducibility, the notebook stores the collection, form, and constraints metadata locally.  
The request below is a template. Check the printed `form_json` and `constraints_json` to confirm the exact allowed field names and values in your CDS account/environment.

Typical use:

1. Authenticate with CDS API.
2. Retrieve UTCI or MRT for a city bbox and date range.
3. Open the downloaded NetCDF/GRIB with xarray.
4. Convert to Zarr for repeated analysis.

In [ ]:
# Uncomment after configuring ~/.cdsapirc or CDS API credentials.
# import cdsapi
#
# client = cdsapi.Client()
#
# city = "Sydney"
# west, south, east, north = CITY_BBOXES[city]
#
# request = {
#     "variable": ["universal_thermal_climate_index"],
#     "product_type": ["consolidated_dataset"],
#     "year": ["2024"],
#     "month": ["01", "02", "12"],
#     "day": [f"{d:02d}" for d in range(1, 32)],
#     "time": ["12:00"],
#     "area": [north, west, south, east],  # north, west, south, east for CDS
#     "data_format": "netcdf",
#     "download_format": "unarchived",
# }
#
# output_nc = DATA_DIR / f"{city.lower()}_utci_2024.nc"
# client.retrieve("derived-utci-historical", request, str(output_nc))
#
# ds = xr.open_dataset(output_nc, chunks={"time": 100})
# ds

## 7. Option B: Open ARCO/Zarr assets if exposed by the related time-series collection

The `derived-utci-historical` collection points to a related collection named **Thermal comfort indices time-series derived from ERA5 reanalysis**.  
If that related collection exposes Zarr assets, they can be opened directly using `xarray.open_zarr`.

In [ ]:
def find_zarr_assets(stac_object: Dict) -> Dict[str, str]:
    """Find potential Zarr assets in a STAC object."""
    out = {}
    for key, asset in stac_object.get("assets", {}).items():
        href = asset.get("href", "")
        asset_type = asset.get("type", "")
        title = asset.get("title", "")
        if "zarr" in href.lower() or "zarr" in asset_type.lower() or "zarr" in title.lower():
            out[key] = href
    return out

if timeseries_collection_href:
    ts_response = requests.get(timeseries_collection_href, timeout=60)
    ts_response.raise_for_status()
    ts_collection = ts_response.json()
    print("Time-series collection ID:", ts_collection.get("id"))
    print("Time-series title:", ts_collection.get("title"))
    print("Assets:", list(ts_collection.get("assets", {}).keys()))
    zarr_assets = find_zarr_assets(ts_collection)
    print("Potential Zarr assets:", zarr_assets)
else:
    ts_collection = None
    zarr_assets = {}
    print("No related time-series collection found in STAC links.")

In [ ]:
# If a Zarr asset is exposed, open it here.
# This cell is intentionally defensive because the public metadata can change.

# Example:
# zarr_href = next(iter(zarr_assets.values()))
# ds = xr.open_zarr(zarr_href, chunks={"time": 365, "latitude": 100, "longitude": 100})
# ds

## 8. Convert UTCI to heat hazard indicators

UTCI is a human thermal comfort index. For tourism heat exposure, useful indicators are:

- `utci_days_above_32`: number of days with strong heat stress.
- `utci_days_above_38`: number of days with very strong heat stress.
- seasonal summaries for summer or peak tourism months.
- city-level and DGGS-cell-level exceedance maps.

This helper works once you have a UTCI `DataArray` from CDS NetCDF, GRIB, or Zarr.

In [ ]:
def infer_coord_names(da: xr.DataArray) -> Tuple[str, str]:
    """Infer latitude and longitude coordinate names."""
    lat_candidates = ["lat", "latitude", "y"]
    lon_candidates = ["lon", "longitude", "x"]
    lat_name = next((c for c in lat_candidates if c in da.coords or c in da.dims), None)
    lon_name = next((c for c in lon_candidates if c in da.coords or c in da.dims), None)
    if lat_name is None or lon_name is None:
        raise ValueError("Could not infer lat/lon coordinate names.")
    return lat_name, lon_name


def to_celsius(da: xr.DataArray) -> xr.DataArray:
    """Convert Kelvin to Celsius if metadata indicates Kelvin."""
    units = str(da.attrs.get("units", "")).lower()
    if units in ["k", "kelvin"] or da.max().compute().item() > 150:
        out = da - 273.15
        out.attrs.update(da.attrs)
        out.attrs["units"] = "degree_Celsius"
        return out
    return da


def compute_utci_heat_indicators(
    utci: xr.DataArray,
    threshold_strong: float = UTCI_STRONG_HEAT,
    threshold_very_strong: float = UTCI_VERY_STRONG_HEAT,
    summer_months: List[int] = [12, 1, 2],
) -> xr.Dataset:
    """Compute annual and summer UTCI threshold exceedance indicators."""
    utci_c = to_celsius(utci)

    annual = xr.Dataset({
        "utci_days_above_32": (utci_c > threshold_strong).groupby("time.year").sum("time"),
        "utci_days_above_38": (utci_c > threshold_very_strong).groupby("time.year").sum("time"),
        "utci_mean": utci_c.groupby("time.year").mean("time"),
        "utci_max": utci_c.groupby("time.year").max("time"),
    })

    summer = utci_c.where(utci_c["time"].dt.month.isin(summer_months), drop=True)
    summer_ds = xr.Dataset({
        "summer_utci_days_above_32": (summer > threshold_strong).groupby("time.year").sum("time"),
        "summer_utci_days_above_38": (summer > threshold_very_strong).groupby("time.year").sum("time"),
        "summer_utci_mean": summer.groupby("time.year").mean("time"),
        "summer_utci_max": summer.groupby("time.year").max("time"),
    })

    return xr.merge([annual, summer_ds])

# Example:
# utci = ds["utci"] or ds["universal_thermal_climate_index"]
# heat_indicators = compute_utci_heat_indicators(utci)
# heat_indicators

## 9. Save climate indicators as Zarr

Zarr is useful here because it stores multi-dimensional climate data in chunks and supports repeated cloud-style analysis.

In [ ]:
# Example after computing indicators:
# zarr_out = OUTPUT_DIR / "sydney_utci_heat_indicators.zarr"
# heat_indicators.chunk({"year": 1}).to_zarr(zarr_out, mode="w")
#
# Re-open later:
# heat_indicators = xr.open_zarr(zarr_out)

## 10. OSM / Overpass API for hotels and tourism amenities

OpenStreetMap is used for tourism exposure layers. The Overpass API can fetch features by bbox and tag.

This query extracts:

### Accommodation
- `tourism=hotel`
- `tourism=hostel`
- `tourism=guest_house`
- `tourism=motel`
- `tourism=apartment`
- `tourism=chalet`

### Attractions and visitor sites
- `tourism=attraction`
- `tourism=museum`
- `tourism=gallery`
- `tourism=viewpoint`
- `tourism=zoo`
- `tourism=theme_park`

### Food and drink
- `amenity=restaurant`
- `amenity=cafe`
- `amenity=bar`
- `amenity=pub`
- `amenity=fast_food`
- `amenity=food_court`

### Transport
- `railway=station`
- `railway=halt`
- `railway=tram_stop`
- `public_transport=station`
- `public_transport=platform`
- `amenity=bus_station`
- `amenity=ferry_terminal`

### Outdoor tourism spaces
- `leisure=park`
- `leisure=garden`
- `leisure=nature_reserve`
- `natural=beach`
- `tourism=camp_site`

### Basic heat-adaptation amenities
- `amenity=drinking_water`
- `amenity=toilets`
- `amenity=shelter`

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

TOURISM_TAGS = "hotel|hostel|guest_house|motel|apartment|chalet|attraction|museum|gallery|viewpoint|zoo|theme_park|camp_site"
AMENITY_TAGS = "restaurant|cafe|bar|pub|fast_food|food_court|bus_station|ferry_terminal|drinking_water|toilets|shelter"
RAILWAY_TAGS = "station|halt|tram_stop|subway_entrance|train_station_entrance"
PUBLIC_TRANSPORT_TAGS = "station|platform|stop_position|stop_area"
LEISURE_TAGS = "park|garden|nature_reserve|swimming_pool"
NATURAL_TAGS = "beach"


def bbox_to_overpass(bbox_wsen: Tuple[float, float, float, float]) -> Tuple[float, float, float, float]:
    """Convert west,south,east,north bbox to Overpass south,west,north,east order."""
    west, south, east, north = bbox_wsen
    return south, west, north, east


def build_tourism_overpass_query(bbox_wsen: Tuple[float, float, float, float]) -> str:
    south, west, north, east = bbox_to_overpass(bbox_wsen)
    bbox_str = f"{south},{west},{north},{east}"
    query = f"""
    [out:json][timeout:180];
    (
      nwr["tourism"~"^({TOURISM_TAGS})$"]({bbox_str});
      nwr["amenity"~"^({AMENITY_TAGS})$"]({bbox_str});
      nwr["railway"~"^({RAILWAY_TAGS})$"]({bbox_str});
      nwr["public_transport"~"^({PUBLIC_TRANSPORT_TAGS})$"]({bbox_str});
      nwr["leisure"~"^({LEISURE_TAGS})$"]({bbox_str});
      nwr["natural"~"^({NATURAL_TAGS})$"]({bbox_str});
    );
    out center tags;
    """
    return query


def fetch_osm_tourism_features(city: str, bbox_wsen: Tuple[float, float, float, float], pause: float = 1.0) -> Dict:
    """Fetch OSM tourism and amenity features for one city bbox using Overpass API."""
    query = build_tourism_overpass_query(bbox_wsen)
    response = requests.post(OVERPASS_URL, data={"data": query}, timeout=240)
    response.raise_for_status()
    data = response.json()

    out_path = DATA_DIR / f"osm_{city.lower()}_tourism_amenities_raw.json"
    with open(out_path, "w") as f:
        json.dump(data, f)

    time.sleep(pause)
    return data

In [ ]:
def osm_element_to_record(element: Dict, city: str) -> Optional[Dict]:
    """Convert one Overpass element to a flat record with point geometry.

    For ways and relations, Overpass returns a centroid-like `center` when using `out center`.
    For exposure analysis this is usually sufficient for POI-style amenities.
    For transport corridors, a more advanced workflow should fetch full geometries.
    """
    tags = element.get("tags", {}) or {}

    if "lat" in element and "lon" in element:
        lat, lon = element["lat"], element["lon"]
    elif "center" in element:
        lat, lon = element["center"].get("lat"), element["center"].get("lon")
    else:
        return None

    if lat is None or lon is None:
        return None

    rec = {
        "city": city,
        "osm_type": element.get("type"),
        "osm_id": element.get("id"),
        "name": tags.get("name"),
        "tourism": tags.get("tourism"),
        "amenity": tags.get("amenity"),
        "railway": tags.get("railway"),
        "public_transport": tags.get("public_transport"),
        "leisure": tags.get("leisure"),
        "natural": tags.get("natural"),
        "website": tags.get("website") or tags.get("contact:website"),
        "operator": tags.get("operator"),
        "addr_full": tags.get("addr:full"),
        "addr_street": tags.get("addr:street"),
        "addr_housenumber": tags.get("addr:housenumber"),
        "lat": float(lat),
        "lon": float(lon),
        "all_tags": json.dumps(tags, ensure_ascii=False),
    }
    rec["geometry"] = Point(float(lon), float(lat))
    return rec


def classify_osm_feature(row: pd.Series) -> str:
    """Assign a project-level tourism exposure category."""
    tourism = row.get("tourism")
    amenity = row.get("amenity")
    railway = row.get("railway")
    public_transport = row.get("public_transport")
    leisure = row.get("leisure")
    natural = row.get("natural")

    if tourism in {"hotel", "hostel", "guest_house", "motel", "apartment", "chalet"}:
        return "accommodation"
    if tourism in {"attraction", "museum", "gallery", "viewpoint", "zoo", "theme_park"}:
        return "attraction"
    if tourism in {"camp_site"}:
        return "outdoor_tourism"
    if amenity in {"restaurant", "cafe", "bar", "pub", "fast_food", "food_court"}:
        return "food_drink"
    if amenity in {"bus_station", "ferry_terminal"} or railway in {"station", "halt", "tram_stop", "subway_entrance", "train_station_entrance"} or public_transport in {"station", "platform", "stop_position", "stop_area"}:
        return "transport"
    if leisure in {"park", "garden", "nature_reserve", "swimming_pool"} or natural in {"beach"}:
        return "outdoor_space"
    if amenity in {"drinking_water", "toilets", "shelter"}:
        return "heat_adaptation_amenity"
    return "other"


def overpass_json_to_gdf(data: Dict, city: str) -> gpd.GeoDataFrame:
    records = []
    for element in data.get("elements", []):
        rec = osm_element_to_record(element, city)
        if rec is not None:
            records.append(rec)

    gdf = gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
    if len(gdf) > 0:
        gdf["category"] = gdf.apply(classify_osm_feature, axis=1)
    return gdf

In [ ]:
# Fetch OSM features for all case-study cities.
# If Overpass is busy, run one city at a time or use a smaller bbox.

osm_gdfs = []

for city, bbox_wsen in CITY_BBOXES.items():
    print(f"Fetching OSM tourism amenities for {city}...")
    data = fetch_osm_tourism_features(city, bbox_wsen)
    gdf = overpass_json_to_gdf(data, city)
    print(city, len(gdf), "features")
    osm_gdfs.append(gdf)

osm_tourism = pd.concat(osm_gdfs, ignore_index=True)
osm_tourism = gpd.GeoDataFrame(osm_tourism, geometry="geometry", crs="EPSG:4326")

# Save as GeoPackage and GeoParquet.
osm_tourism.to_file(OUTPUT_DIR / "osm_tourism_amenities.gpkg", layer="tourism_amenities", driver="GPKG")
osm_tourism.to_parquet(OUTPUT_DIR / "osm_tourism_amenities.parquet", index=False)

osm_tourism.head()

## 11. Quick OSM quality check

OSM completeness varies by city and category. Always inspect category counts before using the data in exposure indicators.

In [ ]:
category_counts = (
    osm_tourism
    .groupby(["city", "category"])
    .size()
    .reset_index(name="count")
    .sort_values(["city", "count"], ascending=[True, False])
)
category_counts

In [ ]:
# Simple plot: tourism feature counts by category
for city in category_counts["city"].unique():
    sub = category_counts[category_counts["city"] == city].sort_values("count", ascending=True)
    ax = sub.plot.barh(x="category", y="count", legend=False, figsize=(8, 5))
    ax.set_title(f"OSM tourism amenities by category: {city}")
    ax.set_xlabel("Feature count")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 12. DGGS / H3 indexing functions

The notebook uses H3 as the example DGGS because it is easy to use from Python and works well as a shared spatial join key.

You can replace H3 with another DGGS later if the project requires a strict OGC DGGS implementation.

In [ ]:
def h3_latlng_to_cell(lat: float, lon: float, resolution: int) -> str:
    """Compatibility wrapper for different h3-py versions."""
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, resolution)
    return h3.geo_to_h3(lat, lon, resolution)


def h3_cell_to_boundary(cell: str):
    """Compatibility wrapper returning boundary as lat/lon pairs."""
    if hasattr(h3, "cell_to_boundary"):
        return h3.cell_to_boundary(cell)
    return h3.h3_to_geo_boundary(cell)


def h3_cell_to_polygon(cell: str) -> Polygon:
    boundary = h3_cell_to_boundary(cell)
    return Polygon([(lon, lat) for lat, lon in boundary])


def point_gdf_to_h3(gdf: gpd.GeoDataFrame, resolution: int, cell_col: str = "h3") -> gpd.GeoDataFrame:
    gdf = gdf.to_crs("EPSG:4326").copy()
    gdf[cell_col] = [
        h3_latlng_to_cell(geom.y, geom.x, resolution)
        for geom in gdf.geometry
        if geom is not None
    ]
    return gdf


def h3_cells_to_gdf(cells: List[str], crs: str = "EPSG:4326") -> gpd.GeoDataFrame:
    unique_cells = sorted(set(cells))
    return gpd.GeoDataFrame(
        {"h3": unique_cells},
        geometry=[h3_cell_to_polygon(c) for c in unique_cells],
        crs=crs,
    )

In [ ]:
# Assign local-level H3 cells to each tourism feature.
osm_tourism_h3 = point_gdf_to_h3(osm_tourism, resolution=H3_LOCAL_RES, cell_col="h3")

# Exposure table: one row per city and H3 cell.
exposure_by_cell = (
    osm_tourism_h3
    .groupby(["city", "h3", "category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Total tourism assets per cell
category_cols = [c for c in exposure_by_cell.columns if c not in ["city", "h3"]]
exposure_by_cell["tourism_asset_count"] = exposure_by_cell[category_cols].sum(axis=1)

exposure_by_cell.to_parquet(OUTPUT_DIR / "osm_tourism_exposure_h3.parquet", index=False)
exposure_by_cell.head()

## 13. Convert climate raster/grid cells to DGGS

This function converts an xarray raster-like `DataArray` into an H3 cell table.

Use it after calculating a heat indicator, for example:

```python
heat_indicator = heat_indicators["summer_utci_days_above_38"].sel(year=2024)
heat_h3 = raster_da_to_h3_table(heat_indicator, resolution=9, value_name="summer_utci_days_above_38")
```

For large rasters, use city subsetting first and/or increase `sample_step`.

In [ ]:
def raster_da_to_h3_table(
    da: xr.DataArray,
    resolution: int,
    value_name: str = "value",
    sample_step: int = 1,
    agg: str = "mean",
) -> pd.DataFrame:
    """Convert a 2D lat/lon DataArray to H3 cell summary table."""
    lat_name, lon_name = infer_coord_names(da)

    da_sub = da.isel({
        lat_name: slice(None, None, sample_step),
        lon_name: slice(None, None, sample_step),
    })

    df = da_sub.to_dataframe(name=value_name).reset_index()
    df = df.dropna(subset=[value_name])

    df["h3"] = [
        h3_latlng_to_cell(float(lat), float(lon), resolution)
        for lat, lon in zip(df[lat_name], df[lon_name])
    ]

    if agg == "mean":
        return df.groupby("h3", as_index=False)[value_name].mean()
    if agg == "max":
        return df.groupby("h3", as_index=False)[value_name].max()
    if agg == "sum":
        return df.groupby("h3", as_index=False)[value_name].sum()

    raise ValueError("agg must be one of: mean, max, sum")

# Example:
# sydney_heat_2024 = heat_indicators["summer_utci_days_above_38"].sel(year=2024)
# sydney_heat_h3 = raster_da_to_h3_table(
#     sydney_heat_2024,
#     resolution=H3_LOCAL_RES,
#     value_name="summer_utci_days_above_38",
#     sample_step=1,
#     agg="mean",
# )

## 14. Join tourism exposure with UTCI heat hazard

Once both climate data and OSM exposure data have an `h3` column, the join is simple.

In [ ]:
# Example placeholder climate-H3 table.
# Replace this with the output of raster_da_to_h3_table from real UTCI data.

# climate_h3 = sydney_heat_h3.copy()
# climate_h3["city"] = "Sydney"

# risk = exposure_by_cell.merge(climate_h3, on=["city", "h3"], how="left")
#
# risk["summer_utci_days_above_38"] = risk["summer_utci_days_above_38"].fillna(0)
# risk["tourism_heat_exposure_score"] = (
#     risk["tourism_asset_count"] * risk["summer_utci_days_above_38"]
# )
#
# risk.sort_values("tourism_heat_exposure_score", ascending=False).head(20)

## 15. Convert risk table to mapped H3 polygons

This creates hexagon polygons for mapping and GIS export.

In [ ]:
def risk_table_to_h3_gdf(risk_df: pd.DataFrame, cell_col: str = "h3") -> gpd.GeoDataFrame:
    cells_gdf = h3_cells_to_gdf(risk_df[cell_col].dropna().unique().tolist())
    out = cells_gdf.merge(risk_df, left_on="h3", right_on=cell_col, how="left")
    return out

# Example:
# risk_gdf = risk_table_to_h3_gdf(risk)
# risk_gdf.to_file(OUTPUT_DIR / "sydney_tourism_heat_risk_h3.gpkg", layer="risk", driver="GPKG")
# risk_gdf.to_parquet(OUTPUT_DIR / "sydney_tourism_heat_risk_h3.parquet", index=False)

## 16. Add Landsat 8 city heat maps

Landsat 8 land surface temperature is useful for identifying intra-urban surface heat differences around tourism spaces.  
It should be treated as a **surface heat layer**, not a direct replacement for UTCI or air temperature.

Suggested workflow:

1. Process Landsat 8 LST for clear-sky scenes.
2. Reproject to EPSG:4326.
3. Convert raster pixels to H3 cells.
4. Join with OSM amenities and UTCI indicators.

In [ ]:
# Example placeholder for Landsat LST if you have a GeoTIFF.
# import rioxarray
#
# landsat_lst = rioxarray.open_rasterio(DATA_DIR / "sydney_landsat8_lst_celsius.tif").squeeze()
# landsat_lst = landsat_lst.rio.reproject("EPSG:4326")
#
# # Rename x/y to lon/lat for helper compatibility
# landsat_lst = landsat_lst.rename({"x": "lon", "y": "lat"})
#
# landsat_h3 = raster_da_to_h3_table(
#     landsat_lst,
#     resolution=H3_LOCAL_RES,
#     value_name="landsat_lst_c",
#     sample_step=5,
#     agg="mean",
# )
# landsat_h3["city"] = "Sydney"

## 17. Add smartphone thermal observations

Smartphone-attached thermal imaging observations can be integrated as point measurements.

Expected columns:

```text
observation_id, timestamp, latitude, longitude, thermal_temperature_c, site_type, shade_status, surface_type
```

In [ ]:
# Example:
# thermal_obs = pd.read_csv(DATA_DIR / "smartphone_thermal_observations.csv")
# thermal_gdf = gpd.GeoDataFrame(
#     thermal_obs,
#     geometry=gpd.points_from_xy(thermal_obs.longitude, thermal_obs.latitude),
#     crs="EPSG:4326",
# )
# thermal_h3 = point_gdf_to_h3(thermal_gdf, resolution=H3_LOCAL_RES, cell_col="h3")
#
# thermal_cell_stats = (
#     thermal_h3
#     .groupby(["city", "h3"])
#     .agg(
#         observed_micro_heat_mean=("thermal_temperature_c", "mean"),
#         observed_micro_heat_max=("thermal_temperature_c", "max"),
#         n_thermal_observations=("thermal_temperature_c", "size"),
#     )
#     .reset_index()
# )

## 18. Final tourism heat exposure score

A simple first version:

```text
tourism_heat_exposure_score = tourism_asset_count × UTCI heat days
```

A richer version can include category weights:

```text
score = heat_hazard × (
    2.0 × accommodation_count +
    1.5 × transport_count +
    1.2 × attraction_count +
    1.0 × food_drink_count +
    0.8 × outdoor_space_count -
    0.5 × heat_adaptation_amenity_count
)
```

This is only a starting point. In the proposal, weights should be justified using tourism vulnerability, exposure duration, outdoor/indoor activity, and local expert input.

In [ ]:
def compute_weighted_tourism_exposure_score(df: pd.DataFrame, heat_col: str) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "accommodation", "transport", "attraction", "food_drink",
        "outdoor_space", "outdoor_tourism", "heat_adaptation_amenity"
    ]:
        if col not in out.columns:
            out[col] = 0

    out[heat_col] = out[heat_col].fillna(0)

    exposure_weight = (
        2.0 * out["accommodation"] +
        1.5 * out["transport"] +
        1.2 * out["attraction"] +
        1.0 * out["food_drink"] +
        1.0 * out["outdoor_tourism"] +
        0.8 * out["outdoor_space"] -
        0.5 * out["heat_adaptation_amenity"]
    )

    out["weighted_tourism_exposure"] = exposure_weight.clip(lower=0)
    out["tourism_heat_exposure_score"] = out["weighted_tourism_exposure"] * out[heat_col]
    return out

# Example:
# risk = compute_weighted_tourism_exposure_score(risk, heat_col="summer_utci_days_above_38")

## 19. Save outputs as GeoParquet and Zarr

Recommended storage pattern:

```text
tourism_heat_cube/
  data/
    cds_derived_utci_historical_collection.json
    osm_sydney_tourism_amenities_raw.json
  outputs/
    utci_heat_indicators.zarr
    osm_tourism_amenities.parquet
    osm_tourism_exposure_h3.parquet
    sydney_tourism_heat_risk_h3.parquet
  stac/
    collection.json
    sydney-tourism-heat-risk-2024.json
```

In [ ]:
# Example save commands after risk table exists:
# risk.to_parquet(OUTPUT_DIR / "tourism_heat_risk_h3.parquet", index=False)
#
# risk_ds = risk.set_index(["city", "h3"]).to_xarray()
# risk_ds.to_zarr(OUTPUT_DIR / "tourism_heat_risk_h3.zarr", mode="w")

## 20. STAC metadata for derived outputs

The derived product can be published as a STAC collection with assets for:

- UTCI heat indicator Zarr cube
- OSM tourism amenities GeoParquet
- DGGS tourism exposure GeoParquet
- final tourism heat risk GeoParquet
- optional Landsat LST COG or Zarr

In [ ]:
derived_stac_item = {
    "type": "Feature",
    "stac_version": "1.1.0",
    "id": "sydney-tourism-heat-exposure-h3-2024",
    "properties": {
        "city": "Sydney",
        "climate_source": "CDS derived-utci-historical / ERA5-HEAT",
        "exposure_source": "OpenStreetMap via Overpass API",
        "dggs": "H3",
        "h3_resolution": H3_LOCAL_RES,
        "heat_thresholds_celsius": [UTCI_STRONG_HEAT, UTCI_VERY_STRONG_HEAT],
        "method": "Raster and vector data are joined through common H3 cell identifiers.",
    },
    "geometry": box(*CITY_BBOXES["Sydney"]).__geo_interface__,
    "bbox": list(CITY_BBOXES["Sydney"]),
    "links": [],
    "assets": {
        "utci_heat_indicators_zarr": {
            "href": "outputs/sydney_utci_heat_indicators.zarr",
            "type": "application/vnd+zarr",
            "title": "UTCI heat indicators as Zarr",
        },
        "osm_tourism_amenities": {
            "href": "outputs/osm_tourism_amenities.parquet",
            "type": "application/x-parquet",
            "title": "OSM tourism amenities from Overpass API",
        },
        "tourism_exposure_h3": {
            "href": "outputs/osm_tourism_exposure_h3.parquet",
            "type": "application/x-parquet",
            "title": "H3-indexed tourism exposure table",
        },
        "tourism_heat_risk_h3": {
            "href": "outputs/tourism_heat_risk_h3.parquet",
            "type": "application/x-parquet",
            "title": "H3-indexed tourism heat exposure risk table",
        },
    },
}

with open(STAC_DIR / "sydney-tourism-heat-exposure-h3-2024.json", "w") as f:
    json.dump(derived_stac_item, f, indent=2)

derived_stac_item

## 21. Proposal methodology text

You can use this directly in the WP1 proposal.

> This work package will implement a STAC-based, Zarr-enabled urban heat and tourism exposure data cube. Thermal comfort indices derived from ERA5 reanalysis, including UTCI, will be accessed through the Copernicus Climate Data Store STAC catalogue and processed into heat hazard indicators such as the number of days exceeding strong and very strong heat stress thresholds. For Brisbane, Gold Coast and Sydney, OpenStreetMap data will be retrieved through the Overpass API to map tourism-relevant exposure layers, including accommodation providers, food and drink venues, museums, attractions, parks, beaches, transport stations and basic heat-adaptation amenities such as drinking water and shelters. A DGGS indexing layer will convert both raster climate data and vector tourism features into common grid-cell identifiers. This enables efficient joining of human thermal comfort data, remote sensing-derived urban heat maps, tourism infrastructure and field-based thermal observations across multiple spatial scales. Final outputs will be stored as analysis-ready Zarr and GeoParquet assets and described using STAC metadata, allowing users to query tourism heat exposure by city, time period, thermal threshold, tourism asset type and grid cell.

## 22. Proposal contribution: why DGGS + Zarr is novel here

The contribution is not just storing data. The important methodological contribution is that **DGGS provides a spatial interoperability layer** between climate, Earth observation and tourism datasets.

In practical terms:

1. **Raster-to-vector problem becomes a cell-index problem.** ERA5 UTCI, Landsat LST, hotels, museums and transport nodes can all be joined by `h3`.
2. **Global and local scales can be connected.** Coarse H3 cells can support global Top 100 city screening, while fine H3 cells support street-level tourism micro-climate mapping.
3. **The workflow is API-ready.** STAC describes the datasets, Zarr stores climate cubes, and GeoParquet stores DGGS-indexed exposure tables.
4. **The workflow is reproducible.** Every derived risk table can be regenerated from CDS STAC metadata, OSM Overpass queries and documented threshold logic.
5. **The outputs can support dashboards.** A web map or dashboard can query pre-computed H3 cells instead of repeatedly processing raw rasters.

## 23. Next technical improvements

Recommended next steps:

1. Add NASA downscaled future projections for 2030–2040s, 2050–2060s and late century.
2. Harmonize all variables into common period/scenario/city dimensions.
3. Add Bureau of Meteorology stations for validation and bias checking.
4. Add Landsat 8 LST processing for city-scale surface heat contrasts.
5. Add smartphone thermal camera observations as micro-climate calibration points.
6. Build a small FastAPI or STAC API endpoint to query by city, bbox, period, threshold and asset type.
7. Publish derived H3 risk tables as PMTiles or vector tiles for dashboard use.